In [0]:
from pyspark.sql import functions as F
energy_usage_df = spark.read.csv(
    "/Workspace/Users/dharshinis369@gmail.com/energy_usage_cleaned.csv",
    header=True,
    inferSchema=True
)
energy_usage_df.printSchema()
energy_usage_df.show(5)
print("The number of rows:",energy_usage_df.count())

root
 |-- log_id: integer (nullable = true)
 |-- device_id: integer (nullable = true)
 |-- device_name: string (nullable = true)
 |-- room_id: integer (nullable = true)
 |-- room_name: string (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- recorded_at: timestamp (nullable = true)

+------+---------+-----------------+-------+-----------+----------+-------------------+
|log_id|device_id|      device_name|room_id|  room_name|energy_kwh|        recorded_at|
+------+---------+-----------------+-------+-----------+----------+-------------------+
|     1|        1|      Ceiling Fan|      1|Living Room|     0.075|2026-10-11 08:00:00|
|     2|        1|      Ceiling Fan|      1|Living Room|     0.075|2026-10-11 20:00:00|
|     3|        2|           LED TV|      1|Living Room|      0.12|2026-10-11 19:00:00|
|     4|        3|Living Room Light|      1|Living Room|     0.015|2026-10-11 18:00:00|
|     5|        5|     Refrigerator|      2|    Kitchen|      0.15|2026-10-11 06:00:0

In [0]:
energy_usage_df = energy_usage_df.withColumn(
    "recorded_at",
    F.to_timestamp("recorded_at")
).withColumn(
    "energy_kwh",
    F.col("energy_kwh").cast("double")
).filter(
    F.col("recorded_at").isNotNull() &
    F.col("energy_kwh").isNotNull()
)

In [0]:
#a daily summary per device
daily_summary = energy_usage_df.groupBy("device_id","device_name",F.to_date("recorded_at").alias("date")).agg(
    F.sum("energy_kwh").alias("total_energy_consumption"),
    F.count("energy_kwh").alias("total_days"),
    F.avg("energy_kwh").alias("avg_daily_consumption"),
    F.max("energy_kwh").alias("max_daily_consumption"),
    F.min("energy_kwh").alias("min_daily_consumption")
).orderBy("device_id","date")
daily_summary.show()

+---------+-----------------+----------+------------------------+----------+---------------------+---------------------+---------------------+
|device_id|      device_name|      date|total_energy_consumption|total_days|avg_daily_consumption|max_daily_consumption|min_daily_consumption|
+---------+-----------------+----------+------------------------+----------+---------------------+---------------------+---------------------+
|        1|      Ceiling Fan|2026-10-11|                    0.15|         2|                0.075|                0.075|                0.075|
|        1|      Ceiling Fan|2026-10-12|                    0.06|         1|                 0.06|                 0.06|                 0.06|
|        1|      Ceiling Fan|2026-10-13|                   0.075|         1|                0.075|                0.075|                0.075|
|        2|           LED TV|2026-10-11|                    0.12|         1|                 0.12|                 0.12|                 0.12|

In [0]:
#a weekly summary of device
weekly_summary = energy_usage_df.groupBy("device_id","device_name",
    F.year("recorded_at").alias("year"),
    F.weekofyear("recorded_at").alias("week")
).agg(
    F.round(F.sum("energy_kwh"), 2).alias("weekly_consumption"),
    F.round(F.avg("energy_kwh"), 2).alias("avg_consumption"),
    F.max("energy_kwh").alias("max_consumption"),
    F.min("energy_kwh").alias("min_consumption")
).orderBy("device_id", "year", "week")

weekly_summary.show()

+---------+-----------------+----+----+------------------+---------------+---------------+---------------+
|device_id|      device_name|year|week|weekly_consumption|avg_consumption|max_consumption|min_consumption|
+---------+-----------------+----+----+------------------+---------------+---------------+---------------+
|        1|      Ceiling Fan|2026|  41|              0.15|           0.08|          0.075|          0.075|
|        1|      Ceiling Fan|2026|  42|              0.14|           0.07|          0.075|           0.06|
|        2|           LED TV|2026|  41|              0.12|           0.12|           0.12|           0.12|
|        2|           LED TV|2026|  42|              0.36|           0.18|           0.24|           0.12|
|        3|Living Room Light|2026|  41|              0.02|           0.02|          0.015|          0.015|
|        3|Living Room Light|2026|  42|              0.02|           0.02|          0.015|          0.015|
|        5|     Refrigerator|2026|  4

In [0]:
#save
daily_summary.write.format("delta").mode("overwrite").saveAsTable("main.default.daily_energy_summary")
weekly_summary.write.format("delta").mode("overwrite").saveAsTable("main.default.weekly_energy_summary")

In [0]:
%sql
select
    device_id,
    device_name,
    avg_daily_consumption,
    max_daily_consumption,
    case
        when avg_daily_consumption > 1.5 then 'High consumption - Reduce usage'
        when avg_daily_consumption > 1 then 'Moderate consumption - Monitor usage'
        else 'Efficient usage'
    end as energy_alert
from main.default.daily_energy_summary
group by device_id, device_name,avg_daily_consumption,max_daily_consumption
order by avg_daily_consumption desc;

device_id,device_name,avg_daily_consumption,max_daily_consumption,energy_alert
16,Bathroom Light,2.02,2.02,High consumption - Reduce usage
9,Air Conditioner,1.2,1.2,Moderate consumption - Monitor usage
9,Air Conditioner,1.175,1.45,Moderate consumption - Monitor usage
9,Air Conditioner,1.1,1.1,Moderate consumption - Monitor usage
17,Water Heater,1.0,1.0,Efficient usage
17,Water Heater,0.8,0.8,Efficient usage
6,Microwave Oven,0.3,0.3,Efficient usage
13,Desktop Computer,0.25,0.25,Efficient usage
13,Desktop Computer,0.24,0.25,Efficient usage
2,LED TV,0.24,0.24,Efficient usage
